# Drug Consumption – Klasyfikacja uzytkownikow LSD

Cel: przewidywanie czy osoba uzywala LSD na podstawie cech demograficznych i osobowosci.
Dane: UCI Drug Consumption Dataset – 1885 respondentow, 11 cech.

In [ ]:
!pip install ucimlrepo -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
%matplotlib inline

## 1. Wczytanie danych

In [ ]:
drug_data = fetch_ucirepo(id=373)
X_raw = drug_data.data.features
y_raw = drug_data.data.targets
df = pd.concat([X_raw, y_raw], axis=1)
print("Ksztalt danych:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Wizualizacja danych

In [ ]:
feature_cols = ["Age", "Gender", "Education", "Country",
                "Nscore", "Escore", "Oscore", "Ascore", "Cscore",
                "Impulsive", "SS"]

df[feature_cols].hist(bins=30, figsize=(20, 12))
plt.suptitle("Rozklad cech wejsciowych", fontsize=16)
plt.tight_layout()
plt.show()

## 3. Analiza korelacji

In [ ]:
# Tworzymy binarna zmienna docelowa: 0 = nigdy nie uzywal LSD, 1 = uzywal
df["LSD_binary"] = (df["LSD"] != "CL0").astype(int)

plt.figure(figsize=(10, 7))
corr = df[feature_cols + ["LSD_binary"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Macierz korelacji")
plt.tight_layout()
plt.show()

## 4. Czyszczenie danych

Semeron to fikcyjny narkotyk w ankiecie – usuwamy osoby ktore zadeklarowaly jego uzycie (nierzetelne odpowiedzi).

In [ ]:
print("Braki danych:")
print(df[feature_cols].isnull().sum())

before = len(df)
df = df[df["Semer"] == "CL0"].copy()
print(f"\nUsunieto {before - len(df)} wierszy. Pozostalo: {len(df)}")

## 5. Przygotowanie danych

In [ ]:
X = df[feature_cols]
y = df["LSD_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Trening: {X_train.shape[0]} probek, Test: {X_test.shape[0]} probek")

## 6. Model 1 – Regresja Logistyczna

In [ ]:
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)
pred_log = log_model.predict(X_test_scaled)

print(classification_report(y_test, pred_log, target_names=["Nie uzywal", "Uzywal"]))

In [ ]:
sns.heatmap(confusion_matrix(y_test, pred_log), annot=True, fmt="d", cmap="Blues")
plt.title("Macierz pomylek - Regresja Logistyczna")
plt.xlabel("Przewidziana")
plt.ylabel("Rzeczywista")
plt.show()

## 7. Model 2 – Random Forest z GridSearchCV

In [ ]:
params = {
    "n_estimators": [50, 100],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    params, cv=3, scoring="f1", n_jobs=-1
)
grid.fit(X_train, y_train)
print("Najlepsze parametry:", grid.best_params_)

In [ ]:
pred_rf = grid.best_estimator_.predict(X_test)
print(classification_report(y_test, pred_rf, target_names=["Nie uzywal", "Uzywal"]))

In [ ]:
sns.heatmap(confusion_matrix(y_test, pred_rf), annot=True, fmt="d", cmap="Greens")
plt.title("Macierz pomylek - Random Forest")
plt.xlabel("Przewidziana")
plt.ylabel("Rzeczywista")
plt.show()

## 8. Waznosc cech

In [ ]:
importance = pd.DataFrame({
    "Cecha": feature_cols,
    "Waznosc": grid.best_estimator_.feature_importances_
}).sort_values("Waznosc", ascending=False)

sns.barplot(data=importance, x="Waznosc", y="Cecha")
plt.title("Waznosc cech - Random Forest")
plt.show()

## 9. Porownanie modeli i wnioski

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

wyniki = pd.DataFrame({
    "Model": ["Regresja Logistyczna", "Random Forest"],
    "Accuracy": [accuracy_score(y_test, pred_log), accuracy_score(y_test, pred_rf)],
    "F1-score": [f1_score(y_test, pred_log), f1_score(y_test, pred_rf)]
})
print(wyniki.to_string(index=False))

wyniki.plot(x="Model", kind="bar", figsize=(8, 4))
plt.title("Porownanie modeli")
plt.xticks(rotation=15)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Wnioski

- **Random Forest osiagnal lepsze wyniki niz Regresja Logistyczna** (wyzsze F1-score i Accuracy).
- Najwazniejsze cechy to **Openness to experience (Oscore)** i **Sensation Seeking (SS)**.
- Dane nie mialy brakow. Usunieto 8 nierzetelnych obserwacji (Semeron).
